In [1]:
# The notebook will need couple module files from code directory.
import os,sys
# os.path.join('..', 'code') is the relative path for code folder, module_path convert it to absolute path, final absolute url: d:\\Study\\Python\\llm-zoomcamp\\code
module_path = os.path.abspath(os.path.join('..', 'code'))

In [2]:
# append the code folder in sys.path if not already exists (100% likely)
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",     #ANN approach
    db_path="../database/faq_vectors2.db"
)

In [6]:
from sentence_transformers import SentenceTransformer

# smallest model from sentence_transformers, 384 dimensions (initial load: 1m)
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
from ingest import load_faq_data
documents=load_faq_data()

In [8]:
texts = []
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [9]:
from tqdm.auto import tqdm
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1368

In [11]:
#vs_index needs vectors and documents (one off process), prepare them before fit function.
#if faq_vector2.db has contents, use clear() to clearn it
vs_index.clear()
vs_index.fit(vectors, documents)

In [16]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(
    query_vector, 
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [17]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [18]:
# remember to close index since it connects db file
vs_index.close()

In [19]:
# reopen vs db
model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="../database/faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(
    query_vector, 
    num_results=5
)
results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO

In [21]:
from rag_helper import RAGVector
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [22]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [23]:
vs_index.close()